In [0]:
# %sql
# CREATE EXTERNAL LOCATION bronze_loc URL 'abfss://bronze@churnlakee22295.dfs.core.windows.net/' WITH (STORAGE CREDENTIAL cred_lake);
# CREATE EXTERNAL LOCATION silver_loc URL 'abfss://silver@churnlakee22295.dfs.core.windows.net/' WITH (STORAGE CREDENTIAL cred_lake);
# CREATE EXTERNAL LOCATION gold_loc   URL 'abfss://gold@churnlakee22295.dfs.core.windows.net/'   WITH (STORAGE CREDENTIAL cred_lake);
# CREATE EXTERNAL LOCATION ml_loc     URL 'abfss://mlartifacts@churnlakee22295.dfs.core.windows.net/' WITH (STORAGE CREDENTIAL cred_lake);


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS churn;
USE CATALOG churn;
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;
CREATE SCHEMA IF NOT EXISTS features;   -- ML feature tables
CREATE SCHEMA IF NOT EXISTS ml;         -- registered models live here

In [0]:
# spark.sql("GRANT USE CATALOG ON CATALOG churn TO `de_engineers`")
# spark.sql("GRANT USE SCHEMA, CREATE TABLE, MODIFY, SELECT ON SCHEMA churn.bronze TO `de_engineers`")
# spark.sql("GRANT USE SCHEMA, CREATE TABLE, MODIFY, SELECT ON SCHEMA churn.silver TO `de_engineers`")
# spark.sql("GRANT USE SCHEMA, CREATE TABLE, MODIFY, SELECT ON SCHEMA churn.gold TO `de_engineers`")
# ## write the raw/curated files
# spark.sql("GRANT READ FILES, WRITE FILES ON EXTERNAL LOCATION bronze_loc TO `de_engineers`")
# spark.sql("GRANT READ FILES, WRITE FILES ON EXTERNAL LOCATION silver_loc TO `de_engineers`")
# spark.sql("GRANT READ FILES, WRITE FILES ON EXTERNAL LOCATION gold_loc TO `de_engineers`")

DataFrame[]

In [0]:
# spark.sql("GRANT USE CATALOG ON CATALOG churn TO `ml_engineers`")
# # read the DE output, but NOT modify it
# spark.sql("GRANT USE SCHEMA, SELECT ON SCHEMA churn.gold TO `ml_engineers`")
# # fully own the ML side
# spark.sql("GRANT USE SCHEMA, CREATE TABLE, MODIFY, SELECT ON SCHEMA churn.features TO `ml_engineers`")
# spark.sql("GRANT USE SCHEMA, CREATE TABLE, CREATE MODEL, CREATE FUNCTION, MODIFY, SELECT ON SCHEMA churn.ml TO `ml_engineers`")
# spark.sql("GRANT READ FILES, WRITE FILES ON EXTERNAL LOCATION ml_loc TO `ml_engineers`")

DataFrame[]

In [0]:
# raw = spark.read.option("header", True).csv("abfss://bronze@churnlakee22295.dfs.core.windows.net/raw/")
# raw.write.format("delta").mode("overwrite").saveAsTable("churn.bronze.customers_raw")

In [0]:
# from pyspark.sql.functions import col, lit, monotonically_increasing_id

# b = spark.table("churn.bronze.customers_raw").dropDuplicates()
# s = (b.withColumn("customer_id", monotonically_increasing_id())
#        .withColumn("age", lit(30))
#        .withColumn("tenure_months", lit(12))
#        .withColumn("monthly_charges", lit(50.0))
#        .withColumn("score", lit(100))
#        .withColumn("name", lit("dummy_name"))
#        .withColumn("num_products", lit(2))
#        .withColumn("is_active", lit(1))
#        .withColumn("churned", lit(0).cast("int")))
# s.write.format("delta").mode("overwrite").saveAsTable("churn.silver.customers")

In [0]:
# g = spark.sql("""
#   SELECT customer_id, age, tenure_months, monthly_charges,
#          num_products, is_active,
#          CAST(churned AS INT) AS churned
#   FROM churn.silver.customers
# """)
# g.write.format("delta").mode("overwrite").saveAsTable("churn.gold.customer_churn")

ML project

In [0]:
%pip install mlflow scikit-learn
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 135.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 621.4/621.4 kB 63.4 MB/s eta 0:00:00
  Attempting uninstall: wcwidth
    Found existing installation: wcwidth 0.2.5
    Not uninstalling wcwidth at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-712a9ccf-0ab0-4dee-9402-2bee38abb200
    Can't uninstall 'wcwidth'. No files were found to uninstall.
  Attempting uninstall: blinker
    Found existing installation: blinker 1.7.0
    Not uninstalling blinker at /usr/lib/python3/dist-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-712a9ccf-0ab0-4dee-9402-2bee38abb200
    Ca

In [0]:
# ml/01_feature_engineering.py
from pyspark.sql.functions import col, when
g = spark.table("churn.gold.customer_churn")
feats = (g
  .withColumn("high_value", when(col("monthly_charges") > 80, 1).otherwise(0))
  .withColumn("tenure_bucket", when(col("tenure_months") < 12, "new")
                               .when(col("tenure_months") < 48, "mid")
                               .otherwise("loyal")))
feats.write.format("delta").mode("overwrite").saveAsTable("churn.features.customer_features")

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
w.workspace.mkdirs("/Shared/churn")          # makes the parent folder

import mlflow
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Shared/churn/experiments")   # now the parent exists

2026/08/16 16:30:21 INFO mlflow.tracking.fluent: Experiment with name '/Shared/churn/experiments' does not exist. Creating a new experiment.
If you are using MLflow Tracing, consider storing your traces in Unity Catalog for unlimited storage (no 100,000 trace limit), fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/4240283707711740', creation_time=1786897821753, effective_trace_archival_retention=None, experiment_id='4240283707711740', last_update_time=1786897821753, lifecycle_stage='active', name='/Shared/churn/experiments', tags={'mlflow.experiment.sourceName': '/Shared/churn/experiments',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'ravi.izp.sharma@avanade.com',
 'mlflow.ownerId': '142404265657761'}, trace_location=None, workspace='default'>

In [0]:
import pandas as pd, numpy as np, mlflow
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

mlflow.sklearn.autolog()   # <-- captures params, metrics, model, signature

df = spark.table("churn.features.customer_features").toPandas()
X = df.drop(columns=["customer_id", "churned", "tenure_bucket"])  # keep numeric simple
y = df["churned"].astype(int)

# Workaround: upstream data has only 1 class and too few rows for demo
if y.nunique() < 2 or len(y) < 10:
    rng = np.random.default_rng(42)
    n_samples = max(len(y), 100)
    X = pd.concat([X] * (n_samples // len(X) + 1), ignore_index=True).head(n_samples)
    y = pd.Series(rng.binomial(1, 0.3, size=n_samples))

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

with mlflow.start_run(run_name="gbm-baseline") as run:
    model = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42)
    model.fit(Xtr, ytr)
    auc = roc_auc_score(yte, model.predict_proba(Xte)[:, 1])
    mlflow.log_metric("test_auc", auc)
    run_id = run.info.run_id
print("AUC:", auc)

2026/08/16 16:34:59 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/local_disk0/.ephemeral_nfs/envs/pythonEnv-712a9ccf-0ab0-4dee-9402-2bee38abb200/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/08/16 16:34:59 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a

AUC: 0.5


In [0]:
mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name="churn.ml.churn_model"
)

Successfully registered model 'churn.ml.churn_model'.
2026/08/16 16:35:47 WARNING mlflow.tracking._model_registry.fluent: Run with id ec7a56d533524537a78d6dcee08349b8 has no artifacts at artifact path 'model', registering model based on models:/m-185000e20d774689b46c18ac977af26f instead


Uploading artifacts:   0%|          | 0/10 [00:00<?, ?it/s]

🔗 Created version '1' of model 'churn.ml.churn_model': https://adb-7405618705938594.14.azuredatabricks.net/explore/data/models/churn/ml/churn_model/version/1?o=7405618705938594


<ModelVersion: aliases=[], creation_timestamp=1786898149485, current_stage=None, deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1786898150766, metrics=[<Metric: dataset_digest='98340161', dataset_name='dataset', key='training_precision_score', model_id='m-185000e20d774689b46c18ac977af26f', run_id='ec7a56d533524537a78d6dcee08349b8', step=0, timestamp=1786898104221, value=0.525625>,
 <Metric: dataset_digest='98340161', dataset_name='dataset', key='training_recall_score', model_id='m-185000e20d774689b46c18ac977af26f', run_id='ec7a56d533524537a78d6dcee08349b8', step=0, timestamp=1786898104221, value=0.725>,
 <Metric: dataset_digest='98340161', dataset_name='dataset', key='training_f1_score', model_id='m-185000e20d774689b46c18ac977af26f', run_id='ec7a56d533524537a78d6dcee08349b8', step=0, timesta

In [0]:
# ml/03_promote_champion.py
from mlflow.tracking import MlflowClient
mlflow.set_registry_uri("databricks-uc")
c = MlflowClient()
c.set_registered_model_alias("churn.ml.churn_model", "champion", version=1)

In [0]:
# ml/04_batch_inference.py
import mlflow, pandas as pd
mlflow.set_registry_uri("databricks-uc")
model = mlflow.pyfunc.load_model("models:/churn.ml.churn_model@champion")

df = spark.table("churn.features.customer_features").toPandas()
X = df.drop(columns=["customer_id", "churned", "tenure_bucket"])
df["churn_score"] = model.predict(X)
spark.createDataFrame(df[["customer_id", "churn_score"]]) \
     .write.format("delta").mode("overwrite").saveAsTable("churn.ml.predictions")

In [0]:
def test_gold_not_empty(spark):
    assert spark.table("churn.gold.customer_churn").count() > 0

def test_primary_key_unique(spark):
    t = spark.table("churn.gold.customer_churn")
    assert t.count() == t.select("customer_id").distinct().count()

def test_no_null_label(spark):
    from pyspark.sql.functions import col
    assert spark.table("churn.gold.customer_churn").filter(col("churned").isNull()).count() == 0

In [0]:
def test_min_auc():
    import mlflow
    mlflow.set_registry_uri("databricks-uc")
    # pull the latest run's AUC and assert a floor
    assert LATEST_AUC >= 0.70, "Model AUC below 0.70 — do not promote"